<a href="https://colab.research.google.com/github/BrianAyekha/-connection-of-MySQL-database-to-NodeJs/blob/main/Week6_Clustering_KMeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6 Assignment | Clustering with K-Means
**Project:** Early fault detection and diagnosis in hospital electromechanical equipment (infant warmer case study)

**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI, id 601), used as a stand-in for infant warmer sensor data.

**Author:** Brian Ayekha Mwangala

## 1. Load the dataset

In [1]:
!pip install -q ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

ai4i = fetch_ucirepo(id=601)
df = pd.concat([ai4i.data.features, ai4i.data.targets], axis=1)
print("Rows, columns:", df.shape)
df.head()

Rows, columns: (10000, 12)


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 2. Select the features to investigate

Clustering is **unsupervised**: the model finds groups without being told the answer. I cluster machines by their operating conditions using the five numeric sensor features:

`Air temperature`, `Process temperature`, `Rotational speed`, `Torque`, `Tool wear`

- `Type` is left out because it is categorical, and K-means works with distances between numbers.
- `Machine failure` and the failure-mode columns are **not** used for clustering. I use them only afterwards to check whether the groups the model finds relate to failures.

In [2]:
features = ["Air temperature", "Process temperature", "Rotational speed", "Torque", "Tool wear"]
X = df[features]

# Check for missing values before clustering
print("Missing values per column:")
print(X.isna().sum())
X.describe().round(2)

Missing values per column:
Air temperature        0
Process temperature    0
Rotational speed       0
Torque                 0
Tool wear              0
dtype: int64


,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
count,10000.0,10000.00,10000.00,10000.00,10000.00
mean,300.0,310.01,1538.78,39.99,107.95
std,2.0,1.48,179.28,9.97,63.65
min,295.3,305.70,1168.00,3.80,0.00
25%,298.3,308.80,1423.00,33.20,53.00
50%,300.1,310.10,1503.00,40.10,108.00
75%,301.5,311.10,1612.00,46.80,162.00
max,304.5,313.80,2886.00,76.60,253.00


## 3. Prepare the features: scaling

K-means groups points by their distance from cluster centres. The features have very different scales (rotational speed is around 1,500 rpm, but torque is around 40 Nm), so rotational speed would dominate the distance. I use **standardisation** (`StandardScaler`) to give every feature a mean of 0 and a standard deviation of 1.

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check: each column should now have mean ~0 and std ~1
print(pd.DataFrame(X_scaled, columns=features).describe().loc[["mean", "std"]].round(3))

      Air temperature  Process temperature  Rotational speed  Torque  \
mean             -0.0                 -0.0              -0.0     0.0   
std               1.0                  1.0               1.0     1.0   

      Tool wear  
mean        0.0  
std         1.0  


## 4. Apply K-means and explore different numbers of clusters

K-means needs the number of clusters *k* to be chosen in advance. I try k = 2 to 10 and compare two measures:
- **Inertia (elbow method):** the total squared distance of points to their cluster centre. It always falls as k grows, so I look for the "elbow" where the improvement slows down.
- **Silhouette score:** how well each point fits its own cluster compared with the nearest other cluster. It ranges from −1 to 1, and higher is better.

In [ ]:
k_values = range(2, 11)
inertias, silhouettes = [], []

for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

results = pd.DataFrame({"k": list(k_values), "Inertia": inertias, "Silhouette": silhouettes})
results.round(3)

In [ ]:
# Plot both measures side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(results["k"], results["Inertia"], marker="o")
axes[0].set_title("Elbow method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(results["k"], results["Silhouette"], marker="o", color="green")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Select the number of clusters

I choose k using both plots. The silhouette score gives a clear number, and the elbow plot shows whether adding more clusters is worth it. The code below picks the k with the highest silhouette score. If the elbow plot suggests a different k that is easier to interpret, I can change `chosen_k` by hand and give my reason in the discussion.

In [ ]:
chosen_k = int(results.loc[results["Silhouette"].idxmax(), "k"])
print("Chosen k:", chosen_k)

# Fit the final model with the chosen k
kmeans = KMeans(n_clusters=chosen_k, n_init=10, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

df["Cluster"].value_counts().sort_index()

## 6. Visualise the clusters

The data has five dimensions, so I use **PCA** to squeeze it into two dimensions for plotting. PCA is used only for the picture. The clustering was done on all five scaled features.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
centres_pca = pca.transform(kmeans.cluster_centers_)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df["Cluster"], cmap="viridis", s=8, alpha=0.6)
plt.scatter(centres_pca[:, 0], centres_pca[:, 1], c="red", marker="X", s=200,
            edgecolor="black", label="Cluster centres")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% of variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% of variance)")
plt.title(f"K-means clusters (k = {chosen_k}) shown in 2D with PCA")
plt.legend(loc="upper right")  # shows the cluster centres marker
plt.grid(alpha=0.3)
plt.show()

## 7. Characteristics of the clusters

In [ ]:
# Average of each feature per cluster, in the original units
profile = df.groupby("Cluster")[features].mean().round(2)
profile["Size"] = df["Cluster"].value_counts().sort_index()
profile["Failure rate (%)"] = (df.groupby("Cluster")["Machine failure"].mean() * 100).round(2)
profile

In [ ]:
# Plain-language description: which features are clearly higher or lower than average?
# z = how many overall standard deviations the cluster mean is from the overall mean
overall_mean, overall_std = X.mean(), X.std()
z = (df.groupby("Cluster")[features].mean() - overall_mean) / overall_std

for c in z.index:
    high = [f for f in features if z.loc[c, f] > 0.5]
    low  = [f for f in features if z.loc[c, f] < -0.5]
    n = (df["Cluster"] == c).sum()
    print(f"Cluster {c} ({n} machines): "
          f"higher than average -> {high if high else 'none'}; "
          f"lower than average -> {low if low else 'none'}")

In [ ]:
# Does any cluster contain more failures? (failure labels were NOT used for clustering)
rates = df.groupby("Cluster")["Machine failure"].mean() * 100
rates.plot(kind="bar", figsize=(6, 4), color="tomato", edgecolor="black")
plt.axhline(df["Machine failure"].mean() * 100, color="black", linestyle="--", label="Overall failure rate")
plt.ylabel("Failure rate (%)")
plt.title("Failure rate in each cluster")
plt.legend()
plt.show()

# Failure modes per cluster
df.groupby("Cluster")[["TWF", "HDF", "PWF", "OSF", "RNF"]].sum()

## 8. Discussion

Read this together with the tables and plots above.

**How I chose k.** The elbow plot shows where adding more clusters stops reducing inertia much, and the silhouette plot shows which k gives the best-separated groups. The code selects the k with the best silhouette score. I confirm that choice against the elbow plot.

**What the groups look like.** The cluster profile table and the printed description show which sensor readings are high or low in each group. In this kind of machine data, I expect groups to be mainly separated by how hard the machine is working (rotational speed and torque) and how worn the tool is. Air and process temperature are narrow and move together, so they are likely to matter less. I need to confirm these expectations against my own output.

**What the clusters might represent.** The clusters could represent different **operating regimes**, for example normal running, high load, and heavily worn or hot-running machines. For an infant warmer, the equivalent would be groups such as idle or low-heat operation, normal control, high heater duty cycle, and an ageing heater. The failure rate chart shows whether any regime carries more risk than the others. If one cluster has a clearly higher failure rate, it could be a warning region to watch. If the rates are similar across clusters, the operating regime alone does not predict failure, and supervised models such as those from Week 5 are needed.

**Limitations.**
- The clusters are patterns in the numbers, not confirmed fault types. Nothing in the data says what they mean, so any labels I give them are my interpretation.
- K-means assumes roughly round clusters of similar size and is sensitive to scaling and to the choice of k.
- The 2D PCA plot hides some of the structure in the five-dimensional data.
- The data is synthetic and from a milling machine, not from infant warmers.

**Next steps.** Try other methods (for example DBSCAN or Gaussian mixtures, which can flag outliers as possible anomalies), add an engineered feature such as the process-minus-air temperature difference, and repeat the analysis on logged data from my warmer prototype.